# Geometry Comparison Tool

Compare molecular geometries against a reference structure.  
Computes **RMSD** and **maximum atomic deviation** after optimal alignment (Kabsch algorithm).  
Supports both plain `.txt` (bare coordinates) and standard `.xyz` format.

---
## Usage
1. Set `REFERENCE_FILE` to your reference geometry.
2. Set `COMPARISON_FILES` to a list of files you want to compare.
3. Run all cells.

In [1]:
import numpy as np
import os

## Configuration — edit here

In [ ]:
# Reference geometry (xyz or plain txt format)
REFERENCE_FILE = "propylene_ccsd_t.txt"

# Files to compare against the reference
COMPARISON_FILES = [
    "propylene_rdm_t.txt",
    "propylene_rdm_t_lambda.txt",
    # add more files here
]

## Parsing

In [3]:
def _is_float(s):
    try:
        float(s)
        return True
    except ValueError:
        return False


def parse_geometry(filepath):
    """
    Parse a geometry file and return (atoms, coords).

    Accepts:
    - xyz format  : first line = atom count, second line = comment, rest = data
    - plain txt   : every line with a symbol + 3 floats is a data line
                    (comment/blank lines are ignored automatically)

    Returns
    -------
    atoms  : list[str]
    coords : np.ndarray  shape (N, 3)
    """
    with open(filepath) as fh:
        raw = [l.rstrip() for l in fh if l.strip()]

    if not raw:
        raise ValueError(f"File is empty: {filepath}")

    atoms, coords = [], []
    first = raw[0].split()

    if len(first) == 1 and first[0].isdigit():
        # xyz format
        expected = int(first[0])
        for line in raw[2:2 + expected]:
            parts = line.split()
            if len(parts) >= 4 and all(_is_float(p) for p in parts[1:4]):
                atoms.append(parts[0])
                coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
    else:
        # plain txt
        for line in raw:
            parts = line.split()
            if len(parts) >= 4 and all(_is_float(p) for p in parts[1:4]):
                atoms.append(parts[0])
                coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
            elif len(parts) == 3 and all(_is_float(p) for p in parts):
                atoms.append("X")
                coords.append([float(parts[0]), float(parts[1]), float(parts[2])])

    if not atoms:
        raise ValueError(f"No coordinate data found in: {filepath}")

    return atoms, np.array(coords, dtype=float)

## Kabsch alignment

In [4]:
def kabsch_rmsd(P, Q):
    """
    Optimally rotate Q onto P (both centroid-centered) using the Kabsch algorithm.
    Returns (rmsd, Q_rotated).
    """
    H = P.T @ Q
    U, S, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    D = np.diag([1.0, 1.0, d])
    R = Vt.T @ D @ U.T
    Q_rot = Q @ R.T
    diff = P - Q_rot
    rmsd = np.sqrt(np.mean(np.sum(diff ** 2, axis=1)))
    return rmsd, Q_rot


def align_and_compare(ref_coords, cmp_coords):
    """
    Center both structures and apply Kabsch rotation.
    Returns (rmsd, max_dev, atom_devs, cmp_aligned).
    """
    P = ref_coords - ref_coords.mean(axis=0)
    Q = cmp_coords - cmp_coords.mean(axis=0)
    rmsd, Q_rot = kabsch_rmsd(P, Q)
    atom_devs = np.linalg.norm(P - Q_rot, axis=1)
    max_dev = atom_devs.max()
    return rmsd, max_dev, atom_devs, Q_rot + ref_coords.mean(axis=0)

## Load reference geometry

In [5]:
ref_atoms, ref_coords = parse_geometry(REFERENCE_FILE)
n_atoms = len(ref_atoms)

print(f"Reference : {REFERENCE_FILE}")
print(f"Atoms     : {n_atoms}  — {' '.join(ref_atoms)}")

Reference : propylene_ccsd_t.txt
Atoms     : 9  — C C C H H H H H H


## Compare each geometry

In [6]:
results = []

for fpath in COMPARISON_FILES:
    print("=" * 65)
    if not os.path.isfile(fpath):
        print(f"SKIPPED (not found): {fpath}")
        continue
    try:
        cmp_atoms, cmp_coords = parse_geometry(fpath)
    except ValueError as exc:
        print(f"SKIPPED (parse error): {fpath}  —  {exc}")
        continue
    if len(cmp_atoms) != n_atoms:
        print(f"SKIPPED (atom count mismatch): {fpath}  [{len(cmp_atoms)} vs {n_atoms}]")
        continue

    rmsd, max_dev, atom_devs, _ = align_and_compare(ref_coords, cmp_coords)
    results.append((fpath, cmp_atoms, rmsd, max_dev, atom_devs))

    print(f"File : {fpath}")
    print(f"  RMSD from reference          : {rmsd:.6f} Å")
    print(f"  Max atomic deviation         : {max_dev:.6f} Å")
    idx = int(np.argmax(atom_devs))
    print(f"  Largest deviation at atom    : {idx + 1} ({cmp_atoms[idx]})  —  {atom_devs[idx]:.6f} Å")
    print(f"  Per-atom deviations (Å):")
    for i, (sym, dev) in enumerate(zip(cmp_atoms, atom_devs)):
        print(f"    Atom {i+1:>3} ({sym:<2}) : {dev:.6f}")

print("=" * 65)

SKIPPED (not found): geom_mp2.xyz
SKIPPED (not found): geom_dft.txt


## Summary table

In [7]:
if not results:
    print("No valid comparison geometries found.")
else:
    results_sorted = sorted(results, key=lambda r: r[2])  # sort by RMSD

    print(f"\n{'File':<45} {'RMSD (Å)':>12} {'Max dev (Å)':>14}")
    print("-" * 73)
    for fpath, _, rmsd, max_dev, _ in results_sorted:
        print(f"{os.path.basename(fpath):<45} {rmsd:>12.6f} {max_dev:>14.6f}")

    best = results_sorted[0]
    print(f"\n>>> Closest geometry to reference : {best[0]}")
    print(f"    RMSD     = {best[2]:.6f} Å")
    print(f"    Max dev  = {best[3]:.6f} Å")

No valid comparison geometries found.


## Bar chart — RMSD comparison

In [ ]:
import matplotlib.pyplot as plt

if results:
    labels = [os.path.basename(r[0]) for r in results_sorted]
    rmsds  = [r[2] for r in results_sorted]
    maxdevs = [r[3] for r in results_sorted]

    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.5), 5))
    bars1 = ax.bar(x - width / 2, rmsds,  width, label="RMSD",        color="steelblue")
    bars2 = ax.bar(x + width / 2, maxdevs, width, label="Max dev",    color="tomato")

    ax.set_xlabel("Geometry file")
    ax.set_ylabel("Deviation (Å)")
    ax.set_title(f"Geometry deviations from {os.path.basename(REFERENCE_FILE)}")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.legend()
    ax.bar_label(bars1, fmt="%.4f", padding=2, fontsize=8)
    ax.bar_label(bars2, fmt="%.4f", padding=2, fontsize=8)
    plt.tight_layout()
    plt.savefig("geom_comparison.png", dpi=150)
    plt.show()
    print("Plot saved to geom_comparison.png")